# Summary
This notebook aims to serve as a quick introduction to building a submission for the AIMO3 competition. It is a refactored version of the notebook that won the 'Early Sharing Prize' for the AIMO2 competition. A key difference is that it outputs a 5-digit final answer, instead of a three-digit final answer for AIMO2. To make the competition more accessible to people new to Kaggle, we provide a large number of comments, explanations, and links to documentation of the libraries used.

Note that this notebook is not designed to achieve a high score on any of the leaderboards, but to serve as a quick start to building your own solution. Therefore, we suggest not simply copying, but adapting and experimenting with parameters, prompting strategies, and so forth.

# Dependencies & Accelerator

### Enabling Acceleratos
To use the provided accelerators, go to `Settings > Accelerator` and select the appropriate GPU to use.

### Dependencies
To install dependencies, we employ a **secondary 'utility' notebook**, which uses `pip install` with `/kaggle/working` as the target directory to pre-install all packages, using spcific versions (frameworks like vLLM typically require latest versions), before the runtime of this notebook is started. Installing packages in this more tedious way is necessary since, for competition submissions, **direct pip installs won't work since the notebook must have the internet turned off** before submitting.

To link a new 'utility' notebook, navigate to the right sidebar and click `Add Input`, then filter by `Your Work` and `Utility Scripts`. Any *public* notebook you have created that you have tagged as "Utility Script" (using `File -> Set as Utility Script` in the 'utility' notebook) should then show up under a "Utility Scripts" tab under the `input` section.

Below are a few small *sanity checks* for the correct torch and numpy versions. They also check if you have enabled a GPU accelerator, as described in the paragraph above.

It is necessary to uninstall a few packages first to avoid conflicts with newer vLLM and numpy versions.

In [1]:
%pip uninstall --yes "tensorflow" "matplotlib" "keras" "scikit-learn"

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: matplotlib 3.7.2
Uninstalling matplotlib-3.7.2:
  Successfully uninstalled matplotlib-3.7.2
Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
assert torch.__version__ == "2.8.0+cu128", (f"Torch version is {torch.__version__} instead of 2.8.0+cu128")
assert torch.cuda.is_available and torch.cuda.device_count() == 1, "GPU not enabled"

In [3]:
import numpy as np
assert np.__version__ == "2.2.0", (f"Numpy version is {np.__version__} instead of 2.2.0")

# Imports

We add the path to the CUDA PTX assembler in order to enable vLLM to compile CUDA graphs as it's in a non-standard location on Kaggle. This results increased throughput :)

In [4]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

In [ ]:
import os
import time
import warnings
import math
import re
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import polars as pl

from transformers import AutoTokenizer, set_seed
from vllm import LLM, SamplingParams
import kaggle_evaluation.aimo_3_inference_server

set_seed(42)
pd.set_option('display.max_colwidth', None)
cutoff_time = time.time() + (4 * 60 + 45) * 60

warnings.simplefilter('ignore')


# Constants

It is good practice to have all constants and configurable parameters that you may change at the top of the file. This allows for quick iteration and changes without scanning the whole notebook each time.

On Kaggle, you will need to download the weights first for this to work. To this end, download them locally and then upload them to Kaggle. Then, you can edit the LLM_MODEL_PATH and input the Kaggle input directory to the weights.

Alternatively, you could also link model weights already uploaded to Kaggle via the Add Input functionality.

In [ ]:
# Path to the downloaded RSFT model weights.
# Update this to your Kaggle model input path before submission.
LLM_MODEL_PATH = 'YOUR_HF_MODEL_PATH_HERE'
TRUST_REMOTE_CODE = True

SYSTEM_PROMPT = "Please reason step by step, and put your final answer within \\boxed{}."
ANSWER_TOKEN = "<ANSWER>"
RETRY_TOKEN = "<RETRY>"
FINALIZE_TOKEN = "<FINALIZE>"

VLLM_ENGINE_KWARGS = {
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.95,
    "dtype": "bfloat16",
    "trust_remote_code": TRUST_REMOTE_CODE,
}

BEST_SELECTOR_PARAMS = {
    "beta_finalize": 1.0,
    "lambda_retry": 0.05,
    "gamma_sampled": 1.0,
    "beam_size": 20,
    "digit_temperature": 1.0,
    "top_p_prefix": None,
    "eps": 1e-8,
}

ROLLOUT_DEFAULTS = {
    "num_of_samples": 120,
    "num_of_retries": 25,
    "z_temperature": 1.2,
    "z_top_p": 0.95,
    "z_min_p": 0.03,
    "z_repetition_penalty": 1.05,
    "digit_temperature": 1.2,
    "digit_top_p": 0.95,
    "digit_min_p": 0.03,
    "answer_bias": -0.5,
}

_Z_TOKEN_RE_LOWER = re.compile(r"^<z_(\d+)>$")
_Z_TOKEN_RE_UPPER = re.compile(r"^<Z_(\d+)>$")


# Loading the Model
### Setting up the environment variables
Each CUDA-enabled device has an ID. Here, we need to set an environment variable using the os package for all devices that should be visible for inference. 0 will enable torch and vLLM to see one GPU with this ID.

In [7]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

### Creating an Inference Engine
---
Use the vLLM model serving engine to load the downloaded weights automatically, specifying the desired precision and other configurations. 
The parameters given for vLLM are documented [here](https://docs.vllm.ai/en/v0.7.2/serving/engine_args.html).
Depending on which model you choose, the default setting may vary, so be sure to read the documentation! 

#### Precision
`dtype="bfloat16"` 

The `dtype` parameter controls the precision with which the weights are loaded. Here, NVIDIA's special half-precision format `bfloat16` is used. 
Which precision can be applied depends on what is available from the weights and on their quantization. Lower precision generally lowers the memory footprint, but will result in decreased accuracy.

#### Maximum Number of Sequences
`max_seq_len=256`

This parameter is specific to vLLM and controls the concurrent requests/prompts that are processed at once. Increasing this will allow the vLLM to fully utilize all GPUs, which is desirable to achieve maximum performance. 
However, larger values naturally come with an increased memory footprint, so there is a risk of running out of memory.

#### Context Length 
`max_model_len=32768` 

An important parameter is the context length, which controls
how many tokens a model will use for its prediction internally.
Below, it is manually set to 32768. If not set, vLLM will
use the model's default configuration instead.

#### GPU Memory Utilization
`gpu_memory_utilization=0.96`

Specifies the fraction of memory vLLM is allowed to reserve per GPU. 
The default of `0.9` is usually sufficient, although higher values
allow the engine to use more memory, possibly fitting larger models,
context windows, concurrent sequences, ...


In [ ]:
def introspect_z_token_ids_and_style(tokenizer) -> Tuple[List[int], str]:
    indexed_lower: List[Tuple[int, int]] = []
    indexed_upper: List[Tuple[int, int]] = []

    def _maybe_add(tok: str, tok_id: int) -> None:
        t = str(tok).strip()
        m_lower = _Z_TOKEN_RE_LOWER.match(t)
        if m_lower is not None:
            indexed_lower.append((int(m_lower.group(1)), int(tok_id)))
            return
        m_upper = _Z_TOKEN_RE_UPPER.match(t)
        if m_upper is not None:
            indexed_upper.append((int(m_upper.group(1)), int(tok_id)))

    try:
        for tok, tok_id in (tokenizer.get_vocab() or {}).items():
            _maybe_add(str(tok), int(tok_id))
    except Exception:
        pass

    try:
        for tok, tok_id in (tokenizer.get_added_vocab() or {}).items():
            _maybe_add(str(tok), int(tok_id))
    except Exception:
        pass

    try:
        dec = getattr(tokenizer, "added_tokens_decoder", {}) or {}
        for tok_id, tok_obj in dec.items():
            tok_text = getattr(tok_obj, "content", None)
            if tok_text is None:
                tok_text = str(tok_obj)
            _maybe_add(str(tok_text), int(tok_id))
    except Exception:
        pass

    if not indexed_lower and not indexed_upper:
        try:
            for tok_id in range(int(len(tokenizer))):
                tok = tokenizer.convert_ids_to_tokens(int(tok_id))
                if tok is None:
                    continue
                _maybe_add(str(tok), int(tok_id))
        except Exception:
            pass

    if indexed_lower:
        indexed_lower = sorted(set(indexed_lower), key=lambda x: x[0])
        return [tok_id for _, tok_id in indexed_lower], "lower"
    if indexed_upper:
        indexed_upper = sorted(set(indexed_upper), key=lambda x: x[0])
        return [tok_id for _, tok_id in indexed_upper], "upper"
    return [], "none"

if LLM_MODEL_PATH == "YOUR_HF_MODEL_PATH_HERE":
    raise ValueError("Set LLM_MODEL_PATH to your RSFT model path before submission.")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_PATH, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise RuntimeError("Tokenizer must have pad_token_id or eos_token_id.")
    tokenizer.pad_token = tokenizer.eos_token

answer_token_id = tokenizer.convert_tokens_to_ids(ANSWER_TOKEN)
retry_token_id = tokenizer.convert_tokens_to_ids(RETRY_TOKEN)
finalize_token_id = tokenizer.convert_tokens_to_ids(FINALIZE_TOKEN)

for tok_text, tok_id in [
    (ANSWER_TOKEN, answer_token_id),
    (RETRY_TOKEN, retry_token_id),
    (FINALIZE_TOKEN, finalize_token_id),
]:
    if tok_id is None or int(tok_id) < 0:
        raise RuntimeError(f"Could not resolve token id for {tok_text!r}.")
    if tokenizer.encode(tok_text, add_special_tokens=False) != [int(tok_id)]:
        raise RuntimeError(f"{tok_text} must map to exactly one token.")

digit_token_ids: List[int] = []
digit_id_to_char: Dict[int, str] = {}
for d in "0123456789":
    ids = tokenizer.encode(d, add_special_tokens=False)
    if len(ids) != 1:
        raise RuntimeError(f"Digit tokenization check failed for {d!r}: got {ids}")
    digit_token_ids.append(int(ids[0]))
    digit_id_to_char[int(ids[0])] = d

z_token_ids, z_style = introspect_z_token_ids_and_style(tokenizer)
if not z_token_ids:
    raise RuntimeError("No Z tokens found in tokenizer (checked <z_i> and <Z_i>).")

z_allowed_token_ids = [int(x) for x in z_token_ids] + [int(answer_token_id)]

llm = LLM(model=LLM_MODEL_PATH, tokenizer=LLM_MODEL_PATH, **VLLM_ENGINE_KWARGS)

print(
    f"Ready. Z style={z_style}, num_z={len(z_token_ids)}, "
    f"answer_id={answer_token_id}, retry_id={retry_token_id}, finalize_id={finalize_token_id}"
)


### Tokenizer

After having created the inference engine (i.e., the `LLM` instance), you will 
also need an appropriate tokenizer for the model. It is necessary to use 
the same tokenizer that the model comes pre-configured with. Changing it will lead to unexpected results. For convenience, vllm allows the loading
of the default tokenizer with the simple command below:

In [ ]:
# Tokenizer and model are initialized above.


### RSFT Selector Configuration

This notebook uses the RSFT inference pipeline with rollout sampling and selector-based decoding.


In [ ]:
def _build_prompt_token_ids(question: str) -> List[int]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(question)},
    ]
    try:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt_text = f"{SYSTEM_PROMPT}\n\n{question}"

    packed = tokenizer(prompt_text, add_special_tokens=False, return_attention_mask=False)
    prompt_ids = [int(x) for x in packed.get("input_ids", [])]
    if not prompt_ids:
        raise RuntimeError("Failed to build prompt token ids.")
    return prompt_ids


def _build_sampling_params(
    *,
    allowed_token_ids: Sequence[int],
    max_tokens: int,
    temperature: float,
    top_p: float,
    min_p: float,
    repetition_penalty: float,
    n: int,
    min_tokens: Optional[int] = None,
    stop_on_answer: bool = False,
    logprobs: Optional[int] = None,
    logit_bias: Optional[Dict[int, float]] = None,
) -> SamplingParams:
    base_kwargs = {
        "max_tokens": int(max_tokens),
        "temperature": float(temperature),
        "top_p": float(top_p),
        "min_p": float(min_p),
        "repetition_penalty": float(repetition_penalty),
        "n": int(n),
    }
    if min_tokens is not None:
        base_kwargs["min_tokens"] = int(min_tokens)
    if stop_on_answer:
        base_kwargs["stop_token_ids"] = [int(answer_token_id)]
    if logprobs is not None:
        base_kwargs["logprobs"] = int(logprobs)
    if logit_bias:
        base_kwargs["logit_bias"] = {int(k): float(v) for k, v in dict(logit_bias).items()}

    allowed = [int(x) for x in allowed_token_ids]
    for keep_min_p in (True, False):
        for keep_logit_bias in (True, False):
            kwargs = dict(base_kwargs)
            if not keep_min_p:
                kwargs.pop("min_p", None)
            if not keep_logit_bias:
                kwargs.pop("logit_bias", None)
            for key, value in (("allowed_token_ids", allowed), ("allowed_token_ids_list", [allowed])):
                try:
                    return SamplingParams(**kwargs, **{key: value})
                except TypeError:
                    continue
    raise RuntimeError("Failed to construct SamplingParams for this vLLM version.")


def _build_verify_probe_sampling_params() -> SamplingParams:
    try:
        return SamplingParams(
            max_tokens=1,
            temperature=1.0,
            top_p=1.0,
            min_p=0.0,
            repetition_penalty=1.0,
            prompt_logprobs=1,
        )
    except TypeError:
        return SamplingParams(
            max_tokens=1,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            prompt_logprobs=1,
        )


# RSFT Utility Functions

The following helpers parse token-level logprobs, normalize distributions, and compute verify-token scores.


In [ ]:
def _extract_logprob_map(entry: Any) -> Dict[int, float]:
    if entry is None:
        return {}
    out: Dict[int, float] = {}
    if isinstance(entry, dict):
        for k, v in entry.items():
            try:
                tok_id = int(k)
            except Exception:
                tok_id = int(getattr(v, "token_id", -1)) if hasattr(v, "token_id") else -1
            if tok_id < 0:
                continue
            if isinstance(v, (int, float)):
                out[tok_id] = float(v)
                continue
            if isinstance(v, dict) and "logprob" in v:
                try:
                    out[tok_id] = float(v["logprob"])
                    continue
                except Exception:
                    pass
            if hasattr(v, "logprob"):
                try:
                    out[tok_id] = float(getattr(v, "logprob"))
                except Exception:
                    pass
        return out
    if isinstance(entry, (list, tuple)):
        for item in entry:
            try:
                out[int(getattr(item, "token_id"))] = float(getattr(item, "logprob"))
            except Exception:
                continue
    return out


def _normalize_probs_from_logprobs(logprob_map: Dict[int, float], allowed_ids: Sequence[int]) -> Tuple[Dict[int, float], Dict[int, float]]:
    allowed = [int(x) for x in allowed_ids]
    vals = [float(logprob_map.get(tok, -1e30)) for tok in allowed]
    max_lp = max(vals)
    exps = [math.exp(v - max_lp) if v > -1e20 else 0.0 for v in vals]
    s = sum(exps)
    if s <= 0.0:
        uni = 1.0 / float(len(allowed))
        probs = {tok: uni for tok in allowed}
        logps = {tok: math.log(uni) for tok in allowed}
        return probs, logps
    probs = {tok: (e / s) for tok, e in zip(allowed, exps)}
    logps = {tok: math.log(max(probs[tok], 1e-300)) for tok in allowed}
    return probs, logps


def _extract_prompt_token_logprob(prompt_logprobs: Any, token_id: int) -> float:
    entries = list(prompt_logprobs or [])
    if len(entries) == 0:
        raise RuntimeError("verify prompt_logprobs payload is empty")
    entry = entries[-1]
    m = _extract_logprob_map(entry)
    if int(token_id) in m:
        return float(m[int(token_id)])
    if hasattr(entry, "token_id") and hasattr(entry, "logprob"):
        try:
            if int(getattr(entry, "token_id")) == int(token_id):
                return float(getattr(entry, "logprob"))
        except Exception:
            pass
    raise RuntimeError(f"prompt_logprobs missing token_id={token_id}; keys={sorted(list(m.keys()))[:10]}")


def _score_verify_token_logprobs_batch(prefix_ids_rows: List[List[int]], verify_probe_params: SamplingParams) -> List[Tuple[float, float]]:
    n = len(prefix_ids_rows)
    if n == 0:
        return []

    prompts_finalize = [{"prompt_token_ids": [int(x) for x in pref] + [int(finalize_token_id)]} for pref in prefix_ids_rows]
    prompts_retry = [{"prompt_token_ids": [int(x) for x in pref] + [int(retry_token_id)]} for pref in prefix_ids_rows]

    outs_finalize = llm.generate(prompts_finalize, verify_probe_params, use_tqdm=False)
    outs_retry = llm.generate(prompts_retry, verify_probe_params, use_tqdm=False)

    if len(outs_finalize) != n or len(outs_retry) != n:
        raise RuntimeError(f"verify batch mismatch finalize={len(outs_finalize)} retry={len(outs_retry)} n={n}")

    out: List[Tuple[float, float]] = []
    for of, or_ in zip(outs_finalize, outs_retry):
        lp_f = _extract_prompt_token_logprob(getattr(of, "prompt_logprobs", None), int(finalize_token_id))
        lp_r = _extract_prompt_token_logprob(getattr(or_, "prompt_logprobs", None), int(retry_token_id))
        out.append((float(lp_f), float(lp_r)))
    return out


#### Answer Selection (Beam Search, Not Majority Vote)

Final prediction is produced by weighted beam search over 5-digit candidates using rollout weights and token probabilities.


In [ ]:
def _collect_rollout_arrays_for_question(
    question: str,
    *,
    num_of_samples: int,
    num_of_retries: int,
    z_temperature: float,
    z_top_p: float,
    z_min_p: float,
    z_repetition_penalty: float,
    digit_temperature: float,
    digit_top_p: float,
    digit_min_p: float,
    answer_bias: float,
) -> Dict[str, np.ndarray]:
    prompt_ids = _build_prompt_token_ids(question)

    z_params = _build_sampling_params(
        allowed_token_ids=z_allowed_token_ids,
        max_tokens=512,
        temperature=float(z_temperature),
        top_p=float(z_top_p),
        min_p=float(z_min_p),
        repetition_penalty=float(z_repetition_penalty),
        n=1,
        stop_on_answer=True,
        logit_bias={int(answer_token_id): float(answer_bias)},
    )
    digit_params = _build_sampling_params(
        allowed_token_ids=digit_token_ids,
        max_tokens=5,
        min_tokens=5,
        temperature=float(digit_temperature),
        top_p=float(digit_top_p),
        min_p=float(digit_min_p),
        repetition_penalty=1.0,
        n=1,
        logprobs=10,
    )
    verify_probe_params = _build_verify_probe_sampling_params()

    active_sequences: List[List[int]] = [[int(x) for x in prompt_ids] for _ in range(int(num_of_samples))]

    retry_list: List[int] = []
    p_finalize_list: List[float] = []
    p_retry_list: List[float] = []
    sampled_digits_list: List[List[int]] = []
    digit_probs_list: List[List[List[float]]] = []

    for retry_idx in range(int(num_of_retries)):
        z_prompts = [{"prompt_token_ids": seq} for seq in active_sequences]
        z_outputs = llm.generate(z_prompts, z_params, use_tqdm=False)

        round_prefixes: List[List[int]] = []
        for base_seq, req_out in zip(active_sequences, z_outputs):
            sample_rows = list(getattr(req_out, "outputs", []) or [])
            if len(sample_rows) != 1:
                raise RuntimeError(f"Expected 1 reasoning sample, got {len(sample_rows)}")
            z_tokens = [int(x) for x in list(getattr(sample_rows[0], "token_ids", []) or [])]
            if int(answer_token_id) in z_tokens:
                cut = z_tokens.index(int(answer_token_id))
                z_tokens = z_tokens[: cut + 1]
            else:
                z_tokens = z_tokens + [int(answer_token_id)]
            round_prefixes.append([int(x) for x in base_seq] + z_tokens)

        digit_prompts = [{"prompt_token_ids": ids} for ids in round_prefixes]
        digit_outputs = llm.generate(digit_prompts, digit_params, use_tqdm=False)

        verify_prefixes: List[List[int]] = []
        next_active: List[List[int]] = []
        pending_digit_probs: List[List[List[float]]] = []
        pending_sampled_digits: List[List[int]] = []

        for prefix_ids, req_out in zip(round_prefixes, digit_outputs):
            sample_rows = list(getattr(req_out, "outputs", []) or [])
            if len(sample_rows) != 1:
                raise RuntimeError(f"Expected 1 digit sample, got {len(sample_rows)}")
            sample = sample_rows[0]
            digit_ids = [int(x) for x in list(getattr(sample, "token_ids", []) or [])]
            if len(digit_ids) != 5:
                raise RuntimeError(f"Expected 5 digit tokens, got {len(digit_ids)}")

            sampled_digits = [int(digit_id_to_char[int(tok)]) for tok in digit_ids]

            lp_payload = list(getattr(sample, "logprobs", []) or [])
            digit_probs_matrix: List[List[float]] = []
            for pos in range(5):
                entry = lp_payload[pos] if pos < len(lp_payload) else None
                lp_map = _extract_logprob_map(entry)
                probs_by_id, _ = _normalize_probs_from_logprobs(lp_map, digit_token_ids)
                row_probs = [float(probs_by_id[tok_id]) for tok_id in digit_token_ids]
                digit_probs_matrix.append(row_probs)

            full_ids = [int(x) for x in prefix_ids] + digit_ids
            verify_prefixes.append(full_ids)
            pending_digit_probs.append(digit_probs_matrix)
            pending_sampled_digits.append(sampled_digits)

            if retry_idx < int(num_of_retries) - 1:
                next_active.append(full_ids + [int(retry_token_id)])
            else:
                next_active.append(full_ids)

        verify_scores = _score_verify_token_logprobs_batch(verify_prefixes, verify_probe_params)
        for sampled_digits, digit_probs_matrix, (lp_f, lp_r) in zip(pending_sampled_digits, pending_digit_probs, verify_scores):
            m = max(lp_f, lp_r)
            zf = math.exp(lp_f - m)
            zr = math.exp(lp_r - m)
            denom = max(zf + zr, 1e-300)
            p_f = float(zf / denom)
            p_r = float(zr / denom)

            retry_list.append(int(retry_idx))
            p_finalize_list.append(p_f)
            p_retry_list.append(p_r)
            sampled_digits_list.append(sampled_digits)
            digit_probs_list.append(digit_probs_matrix)

        active_sequences = next_active

    out = {
        "retry_index": np.asarray(retry_list, dtype=np.int64),
        "p_finalize": np.asarray(p_finalize_list, dtype=np.float64),
        "p_retry": np.asarray(p_retry_list, dtype=np.float64),
        "sampled_digits": np.asarray(sampled_digits_list, dtype=np.int64),
        "digit_probs": np.asarray(digit_probs_list, dtype=np.float64),
    }

    expected = int(num_of_samples) * int(num_of_retries)
    if out["digit_probs"].shape != (expected, 5, 10):
        raise RuntimeError(f"digit_probs shape mismatch: {out['digit_probs'].shape} expected {(expected, 5, 10)}")

    return out


#### Logprob and Probability Helpers

Utilities convert vLLM logprob payloads into stable probability tables over digits.


In [ ]:
def _apply_digit_temperature(probs: np.ndarray, digit_temperature: float) -> np.ndarray:
    t = float(digit_temperature)
    if t <= 0.0:
        raise ValueError("digit_temperature must be > 0")
    if abs(t - 1.0) < 1e-12:
        return probs

    clipped = np.clip(probs, 1e-300, 1.0)
    adjusted = np.power(clipped, 1.0 / t)
    adjusted = adjusted / np.clip(adjusted.sum(axis=2, keepdims=True), 1e-300, None)
    return adjusted


def _compute_rollout_weights(
    retry_index: np.ndarray,
    p_finalize: np.ndarray,
    p_retry: np.ndarray,
    beta_finalize: float,
    lambda_retry: float,
    eps: float,
) -> np.ndarray:
    ratio = p_finalize / np.maximum(p_retry, float(eps))
    w = np.exp(-float(lambda_retry) * retry_index.astype(np.float64)) * np.power(ratio, float(beta_finalize))
    return np.clip(w, 1e-300, 1e300)


def _logsumexp(x: np.ndarray) -> float:
    m = float(np.max(x))
    if not np.isfinite(m):
        return float("-inf")
    return float(m + math.log(float(np.sum(np.exp(x - m)))))


def _score_prefix(
    prefix: Tuple[int, ...],
    *,
    weights: np.ndarray,
    log_weights: np.ndarray,
    log_digit_probs: np.ndarray,
    sampled_digits: np.ndarray,
    gamma_sampled: float,
) -> Tuple[float, float, float]:
    k = len(prefix)
    idx = np.arange(k)
    pref = np.asarray(prefix, dtype=np.int64)

    log_prefix_prob = log_digit_probs[:, idx, pref].sum(axis=1)
    log_terms = log_weights + log_prefix_prob
    log_s_soft = _logsumexp(log_terms)
    s_soft = math.exp(min(log_s_soft, 700.0))

    sampled_match = np.all(sampled_digits[:, :k] == pref[None, :], axis=1)
    s_sampled = float(np.sum(weights[sampled_match]))
    s_total = float(s_soft + float(gamma_sampled) * s_sampled)
    return s_total, s_soft, s_sampled


def _prune_candidates(
    candidates: List[Tuple[Tuple[int, ...], float, float, float]],
    beam_size: int,
    top_p_prefix: Optional[float],
) -> List[Tuple[Tuple[int, ...], float, float, float]]:
    ordered = sorted(candidates, key=lambda x: (-x[1], -x[3], x[0]))

    if top_p_prefix is not None:
        tp = float(top_p_prefix)
        if tp <= 0.0 or tp > 1.0:
            raise ValueError("top_p_prefix must be in (0,1]")
        scores = np.asarray([max(float(x[1]), 0.0) for x in ordered], dtype=np.float64)
        total = float(scores.sum())
        if total > 0.0:
            probs = scores / total
            cdf = np.cumsum(probs)
            keep_n = int(np.searchsorted(cdf, tp, side="left") + 1)
            ordered = ordered[:keep_n]

    return ordered[:int(beam_size)]


#### Verify-Token Scoring

We score `<FINALIZE>` vs `<RETRY>` token likelihoods to weight each rollout.


In [ ]:
def _select_one_answer_from_rollouts(
    *,
    retry_index: np.ndarray,
    p_finalize: np.ndarray,
    p_retry: np.ndarray,
    sampled_digits: np.ndarray,
    digit_probs: np.ndarray,
    beta_finalize: float,
    lambda_retry: float,
    gamma_sampled: float,
    beam_size: int,
    digit_temperature: float,
    top_p_prefix: Optional[float],
    eps: float,
) -> str:
    weights = _compute_rollout_weights(
        retry_index=retry_index,
        p_finalize=p_finalize,
        p_retry=p_retry,
        beta_finalize=float(beta_finalize),
        lambda_retry=float(lambda_retry),
        eps=float(eps),
    )
    log_weights = np.log(np.clip(weights, 1e-300, None))

    probs = _apply_digit_temperature(digit_probs, float(digit_temperature))
    log_digit_probs = np.log(np.clip(probs, 1e-300, 1.0))

    beam: List[Tuple[Tuple[int, ...], float, float, float]] = []
    for _ in range(5):
        parents = beam if len(beam) > 0 else [(tuple(), 0.0, 0.0, 0.0)]
        candidates: List[Tuple[Tuple[int, ...], float, float, float]] = []
        for pref, _, _, _ in parents:
            for d in range(10):
                new_pref = tuple(list(pref) + [int(d)])
                total, s_soft, s_sampled = _score_prefix(
                    new_pref,
                    weights=weights,
                    log_weights=log_weights,
                    log_digit_probs=log_digit_probs,
                    sampled_digits=sampled_digits,
                    gamma_sampled=float(gamma_sampled),
                )
                candidates.append((new_pref, total, s_soft, s_sampled))
        beam = _prune_candidates(candidates, beam_size=int(beam_size), top_p_prefix=top_p_prefix)

    if len(beam) == 0:
        raise RuntimeError("Beam is empty; cannot decode final answer")

    best_pref = beam[0][0]
    return "".join(str(int(x)) for x in best_pref)


def predict_final_answer(question: str) -> str:
    roll = _collect_rollout_arrays_for_question(
        question=question,
        num_of_samples=int(ROLLOUT_DEFAULTS["num_of_samples"]),
        num_of_retries=int(ROLLOUT_DEFAULTS["num_of_retries"]),
        z_temperature=float(ROLLOUT_DEFAULTS["z_temperature"]),
        z_top_p=float(ROLLOUT_DEFAULTS["z_top_p"]),
        z_min_p=float(ROLLOUT_DEFAULTS["z_min_p"]),
        z_repetition_penalty=float(ROLLOUT_DEFAULTS["z_repetition_penalty"]),
        digit_temperature=float(ROLLOUT_DEFAULTS["digit_temperature"]),
        digit_top_p=float(ROLLOUT_DEFAULTS["digit_top_p"]),
        digit_min_p=float(ROLLOUT_DEFAULTS["digit_min_p"]),
        answer_bias=float(ROLLOUT_DEFAULTS["answer_bias"]),
    )

    return _select_one_answer_from_rollouts(
        retry_index=roll["retry_index"],
        p_finalize=roll["p_finalize"],
        p_retry=roll["p_retry"],
        sampled_digits=roll["sampled_digits"],
        digit_probs=roll["digit_probs"],
        beta_finalize=float(BEST_SELECTOR_PARAMS["beta_finalize"]),
        lambda_retry=float(BEST_SELECTOR_PARAMS["lambda_retry"]),
        gamma_sampled=float(BEST_SELECTOR_PARAMS["gamma_sampled"]),
        beam_size=int(BEST_SELECTOR_PARAMS["beam_size"]),
        digit_temperature=float(BEST_SELECTOR_PARAMS.get("digit_temperature", 1.0)),
        top_p_prefix=BEST_SELECTOR_PARAMS.get("top_p_prefix", None),
        eps=float(BEST_SELECTOR_PARAMS.get("eps", 1e-8)),
    )


# Rollout Collection

Collects reasoning-token rollouts, digit completions, and verify scores across retries.


In [ ]:
# RSFT selector pipeline is defined in the cells above.


#### Prompt + Sampling Construction

Prompt tokenization and constrained sampling params are configured for Z-tokens and digits.


In [ ]:
# RSFT selector pipeline is defined in the cells above.


### Rollout Assembly

Each retry appends reasoning tokens, samples 5 digits, and records verify probabilities.


In [ ]:
# RSFT selector pipeline is defined in the cells above.


### Beam Scoring and Pruning

Candidate prefixes are scored with soft probability mass plus sampled support, then pruned by beam size.


In [ ]:
# RSFT selector pipeline is defined in the cells above.


# Inference

`predict_for_question` runs the RSFT selector end-to-end and returns an integer answer in `[0, 99999]`.


In [ ]:
# RSFT selector pipeline is defined in the cells above.


### Predicting a Single Problem

This wraps the selector and applies a safe fallback if inference fails or time is exceeded.


In [ ]:
def predict_for_question(question: str) -> int:
    if time.time() > cutoff_time:
        return 210

    try:
        answer_text = predict_final_answer(question)
        answer_int = int(answer_text)
        if 0 <= answer_int <= 99999:
            return answer_int
    except Exception as exc:
        print(f"predict_for_question failed: {exc}")

    return 210


# Setup for Kaggle

### The predict function
Kaggles inference server works by passing in a `predict` function with a specific 
signature that handles predictions for a single problem.

The function must fit exactly the signature, as given below, and the following must hold:

* The function should **return a single integer** between **0 and 99999**, inclusive.
* Take care that each call of `predict` returns a final answer **within the allotted time**.

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    row_id = id_.item(0)
    question_str = question.item(0)
    pred = predict_for_question(question_str)
    return pl.DataFrame({'id': row_id, 'answer': pred})


### Starting the inference server
Kaggle provides a package on their platform called `kaggle_evaluation` which allows
users to connect to a remote or spawn a local inference server for a competition.
The inference server is initialized with our `predict` function as a parameter.

To run problems from a file, it uses `run_local_gateway` (if local) or serve (when connecting to a remote) to receive the input problems. You will not have to worry about this part, as Kaggle will provide the code for it in any case.

If running locally, take care that the CSV used as input only contains the columns
`id,problem`. You will have to manually remove any other columns that may be used for
analysis of your solution (answers, metadata, ...).

In [22]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

------
111bbb
What is $0\times10$?
round 0


Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{'role': 'assistant', 'content': '<think>\nOkay, so I need to figure out what 0 multiplied by 10 is. Hmm, let me think. I remember that multiplying by zero is a special case in math. Let me start from the basics. Multiplication is basically repeated addition, right? So if I have 0 times 10, that means I\'m adding 10 zero times. But wait, if I add 10 zero times, what does that mean? Does that mean I have nothing? Like, if I add 10 one time, it\'s 10. If I add it two times, it\'s 10 + 10 = 20. So adding it zero times... would that just be zero? Because I didn\'t add anything?\n\nBut maybe there\'s another way to think about it. What if I reverse it? 10 times 0. That might be easier for me. Ten groups of zero. If I have ten groups, each containing zero elements, then the total number of elements is zero in each group, so adding them all up would still be zero. Yeah, that makes sense. So 10 times 0 is 0. And since multiplication is commutative, meaning the order doesn\'t matter, 0 times 10

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{'role': 'assistant', 'content': "<think>\nOkay, so I need to figure out what 1 minus 1 is. Let me start by recalling basic subtraction. Subtraction is taking one number away from another. Here, we have 1 minus 1. That means we start with 1 and take away 1. \n\nHmm, if I have one apple and I take one apple away, how many apples are left? None, right? So maybe the answer is 0? Wait, but let me make sure I'm not missing something here. Sometimes in math, things can be tricky, but this seems straightforward.\n\nLet me think about the number line. If I start at 1 and move one unit to the left, where do I land? That should be 0. Yeah, that makes sense. Also, if I have the equation 1 - 1, it's like adding the additive inverse. The additive inverse of 1 is -1, so 1 + (-1) equals 0. That checks out too.\n\nAnother way to look at it is using properties of numbers. The subtraction operation is defined as adding the opposite. So, subtracting a number is the same as adding its negative. Therefore,

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{'role': 'assistant', 'content': "<think>\nOkay, so I need to solve the equation 4 plus x equals 4. Hmm, let me think. Alright, the equation is 4 + x = 4. I need to find the value of x that makes this equation true. \n\nFirst, I remember that when solving for a variable, the goal is to isolate it on one side of the equation. In this case, the variable is x, and it's currently being added to 4. To get x by itself, I should do the opposite operation. Since it's addition, the opposite operation would be subtraction. \n\nSo, if I subtract 4 from both sides of the equation, that should help isolate x. Let me write that down. If I subtract 4 from the left side, it would be 4 + x - 4. And I have to do the same to the right side, which is 4 - 4. \n\nSimplifying the left side, 4 - 4 cancels out, leaving just x. On the right side, 4 - 4 is also 0. So that gives me x = 0. \n\nWait, let me check if that makes sense. If x is 0, then plugging it back into the original equation: 4 + 0 equals 4. Yeah,

# Reference
As mentioned above, this notebook is based on the notebook by Md Boktiar Mahbub Murad submitted for the AIMO2 'Early Sharing Prize'. To make the notebook a bit more approachable, we cleaned it and adding copious comments & documentation for the
contestants of AIMO3.